In [0]:
# ============================================================
# CELLULE 0 — CHEMINS
# Toujours en premier absolu
# ============================================================
PATH_AI4I     = '/Volumes/workspace/default/bronze/ai4i2020.csv'
PATH_MAINT    = '/Volumes/workspace/default/bronze/predictive_maintenance_v3.csv'
PATH_SILVER   = '/Volumes/workspace/default/bronze/silver/sensor_clean/'
PATH_FEATURES = '/Volumes/workspace/default/bronze/silver/sensor_features/'

print('✅ Chemins OK')

✅ Chemins OK


In [0]:
# ============================================================
# CELLULE 1 — IMPORTS
# Toujours avant toute lecture de données
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

print('✅ Imports PySpark OK')

✅ Imports PySpark OK


In [0]:
# ============================================================
# CELLULE 2 — LECTURE SILVER
# On repart du Silver propre produit en J5
# Jamais on retouche au Bronze — règle d'or
# ============================================================
df = spark.read.parquet(PATH_SILVER)

print(f'Lignes Silver chargées : {df.count()}')
print(f'Colonnes disponibles   : {len(df.columns)}')
print()
print('Liste des colonnes :')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2}. {col}')

df.show(3)

Lignes Silver chargées : 10000
Colonnes disponibles   : 15

Liste des colonnes :
   1. sensor_id
   2. product_id
   3. station_type
   4. air_temp_k
   5. process_temp_k
   6. rotation_speed_rpm
   7. torque_nm
   8. tool_wear_min
   9. machine_failure
  10. tool_wear_failure
  11. heat_dissipation_failure
  12. power_failure
  13. overstrain_failure
  14. random_failure
  15. timestamp
+---------+----------+------------+----------+--------------+------------------+---------+-------------+---------------+-----------------+------------------------+-------------+------------------+--------------+-------------------+
|sensor_id|product_id|station_type|air_temp_k|process_temp_k|rotation_speed_rpm|torque_nm|tool_wear_min|machine_failure|tool_wear_failure|heat_dissipation_failure|power_failure|overstrain_failure|random_failure|          timestamp|
+---------+----------+------------+----------+--------------+------------------+---------+-------------+---------------+-----------------+-------

In [0]:
# ============================================================
# CELLULE 3 — DÉFINITION DES FENÊTRES GLISSANTES
#
# Window.orderBy('sensor_id') = on trie par ordre chronologique
# rowsBetween(-60, 0)  = les 60 lignes précédentes + ligne actuelle
# rowsBetween(-180, 0) = les 180 lignes précédentes + ligne actuelle
#
# Chaque ligne = 1 mesure par minute → 60 lignes = 1 heure
# ============================================================
window_1h = Window.orderBy('sensor_id').rowsBetween(-60, 0)
window_3h = Window.orderBy('sensor_id').rowsBetween(-180, 0)

print('Fenêtres définies :')
print('  window_1h → 60 mesures précédentes  (≈ 1 heure)')
print('  window_3h → 180 mesures précédentes (≈ 3 heures)')

Fenêtres définies :
  window_1h → 60 mesures précédentes  (≈ 1 heure)
  window_3h → 180 mesures précédentes (≈ 3 heures)


In [0]:
# ============================================================
# CELLULE 4 — ROLLING FEATURES TEMPÉRATURE
#
# rolling_mean_temp_1h : si cette moyenne monte → surchauffe imminente
# rolling_std_temp_1h  : si l'écart-type explose → instabilité thermique
# ============================================================
df = df \
    .withColumn('rolling_mean_temp_1h',
        F.avg('process_temp_k').over(window_1h)) \
    .withColumn('rolling_std_temp_1h',
        F.stddev('process_temp_k').over(window_1h))

print('✅ Rolling features température créées')
df.select('sensor_id', 'process_temp_k',
          'rolling_mean_temp_1h', 'rolling_std_temp_1h').show(8)

✅ Rolling features température créées


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+--------------+--------------------+-------------------+
|sensor_id|process_temp_k|rolling_mean_temp_1h|rolling_std_temp_1h|
+---------+--------------+--------------------+-------------------+
|        1|         308.6|               308.6|               NULL|
|        2|         308.7|              308.65|0.07071067811863063|
|        3|         308.5|  308.59999999999997|0.09999999999998012|
|        4|         308.6|               308.6|0.08164965809275636|
|        5|         308.7|              308.62|0.08366600265340111|
|        6|         308.6|  308.61666666666673|0.07527726527089905|
|        7|         308.6|  308.61428571428576|0.06900655593422471|
|        8|         308.6|            308.6125|0.06408699444615465|
+---------+--------------+--------------------+-------------------+
only showing top 8 rows


In [0]:
# ============================================================
# CELLULE 5 — ROLLING FEATURES VITESSE DE ROTATION
#
# rolling_mean_rotation_1h : chute = problème mécanique
# rolling_std_rotation_1h  : hausse = vibrations anormales
# ============================================================
df = df \
    .withColumn('rolling_mean_rotation_1h',
        F.avg('rotation_speed_rpm').over(window_1h)) \
    .withColumn('rolling_std_rotation_1h',
        F.stddev('rotation_speed_rpm').over(window_1h))

print('✅ Rolling features rotation créées')
df.select('sensor_id', 'rotation_speed_rpm',
          'rolling_mean_rotation_1h', 'rolling_std_rotation_1h').show(8)

✅ Rolling features rotation créées


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+------------------+------------------------+-----------------------+
|sensor_id|rotation_speed_rpm|rolling_mean_rotation_1h|rolling_std_rotation_1h|
+---------+------------------+------------------------+-----------------------+
|        1|            1551.0|                  1551.0|                   NULL|
|        2|            1408.0|                  1479.5|      101.1162697096763|
|        3|            1498.0|      1485.6666666666667|      72.29338374521788|
|        4|            1433.0|                  1472.5|      64.63487190879756|
|        5|            1408.0|                  1459.6|      62.97062807372975|
|        6|            1425.0|      1453.8333333333333|      58.06691542235275|
|        7|            1558.0|      1468.7142857142858|      66.02957490325653|
|        8|            1527.0|                  1476.0|      64.51135005341534|
+---------+------------------+------------------------+-----------------------+
only showing top 8 rows


In [0]:
# ============================================================
# CELLULE 6 — ROLLING FEATURES COUPLE (TORQUE)
#
# On calcule sur 1h ET 3h pour capturer des tendances courtes et longues
# Si rolling_3h monte mais rolling_1h reste stable → dégradation lente
# Si les deux montent vite → dégradation rapide, panne imminente
# ============================================================
df = df \
    .withColumn('rolling_mean_torque_1h',
        F.avg('torque_nm').over(window_1h)) \
    .withColumn('rolling_mean_torque_3h',
        F.avg('torque_nm').over(window_3h))

print('✅ Rolling features couple créées')
df.select('sensor_id', 'torque_nm',
          'rolling_mean_torque_1h', 'rolling_mean_torque_3h').show(8)

✅ Rolling features couple créées


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+---------+----------------------+----------------------+
|sensor_id|torque_nm|rolling_mean_torque_1h|rolling_mean_torque_3h|
+---------+---------+----------------------+----------------------+
|        1|     42.8|                  42.8|                  42.8|
|        2|     46.3|                 44.55|                 44.55|
|        3|     49.4|    46.166666666666664|    46.166666666666664|
|        4|     39.5|                  44.5|                  44.5|
|        5|     40.0|                  43.6|                  43.6|
|        6|     41.9|     43.31666666666666|     43.31666666666666|
|        7|     42.4|    43.185714285714276|    43.185714285714276|
|        8|     40.2|     42.81249999999999|     42.81249999999999|
+---------+---------+----------------------+----------------------+
only showing top 8 rows


In [0]:
# ============================================================
# CELLULE 7 — DIFFÉRENTIEL DE TEMPÉRATURE
#
# temp_diff = process_temp - air_temp
# Normalement : la température process est toujours > air_temp
# Si l'écart se réduit → le système de refroidissement lâche
# ============================================================
df = df.withColumn(
    'temp_diff',
    F.col('process_temp_k') - F.col('air_temp_k')
)

print('✅ temp_diff créé')
df.select('sensor_id', 'air_temp_k', 'process_temp_k', 'temp_diff') \
  .orderBy('temp_diff') \
  .show(5)

print('\nStatistiques temp_diff :')
df.select('temp_diff').describe().show()

✅ temp_diff créé


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+----------+--------------+-----------------+
|sensor_id|air_temp_k|process_temp_k|        temp_diff|
+---------+----------+--------------+-----------------+
|     4421|     302.6|         310.2|7.599999999999966|
|     4424|     302.6|         310.2|7.599999999999966|
|     4422|     302.6|         310.2|7.599999999999966|
|     4420|     302.6|         310.2|7.599999999999966|
|     4423|     302.6|         310.2|7.599999999999966|
+---------+----------+--------------+-----------------+
only showing top 5 rows

Statistiques temp_diff :
+-------+------------------+
|summary|         temp_diff|
+-------+------------------+
|  count|             10000|
|   mean|10.000629999999948|
| stddev|1.0010938127779077|
|    min| 7.599999999999966|
|    max|12.100000000000023|
+-------+------------------+



In [0]:
  # ============================================================
  # CELLULE 8 — PUISSANCE ESTIMÉE
  #
  # Formule physique : P (watts) = Couple (Nm) × Vitesse angulaire (rad/s)
  # Vitesse angulaire = RPM × 2π / 60
  # Donc : P = torque_nm × rotation_speed_rpm × 2π / 60
  #
  # Signal : une chute de puissance sans baisse de vitesse
  # = perte d'efficacité mécanique = usure ou défaillance
  # ============================================================
  df = df.withColumn(
      'power_estimated_w',
      (F.col('torque_nm') * F.col('rotation_speed_rpm') * 2 * 3.14159) / 60
  )

  print('✅ power_estimated_w créé')
  df.select('sensor_id', 'torque_nm', 'rotation_speed_rpm', 'power_estimated_w') \
    .show(5)

  print('\nStatistiques puissance estimée :')
  df.select('power_estimated_w').describe().show()

✅ power_estimated_w créé


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+---------+------------------+-----------------+
|sensor_id|torque_nm|rotation_speed_rpm|power_estimated_w|
+---------+---------+------------------+-----------------+
|        1|     42.8|            1551.0|6951.584688399999|
|        2|     46.3|            1408.0|6826.716957866665|
|        3|     49.4|            1498.0|7749.380996933333|
|        4|     39.5|            1433.0|5927.499652166666|
|        5|     40.0|            1408.0|5897.811626666667|
+---------+---------+------------------+-----------------+
only showing top 5 rows

Statistiques puissance estimée :
+-------+------------------+
|summary| power_estimated_w|
+-------+------------------+
|  count|             10000|
|   mean| 6264.673704363829|
| stddev|1093.0257510180215|
|    min| 2612.729503416668|
|    max| 9755.987833699997|
+-------+------------------+



In [0]:
# ============================================================
# CELLULE 9 — TAUX D'USURE DE L'OUTIL
#
# tool_wear_rate = usure_cumulée / nombre_de_mesures_écoulées
# On ajoute +1 au dénominateur pour éviter division par zéro
#
# Signal : si ce taux accélère dans le temps
# → l'outil se dégrade plus vite qu'il ne devrait
# → maintenance préventive nécessaire
# ============================================================
df = df.withColumn(
    'tool_wear_rate',
    F.col('tool_wear_min') / (F.col('sensor_id') + 1)
)

print('✅ tool_wear_rate créé')
df.select('sensor_id', 'tool_wear_min', 'tool_wear_rate').show(8)

✅ tool_wear_rate créé


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-------------+------------------+
|sensor_id|tool_wear_min|    tool_wear_rate|
+---------+-------------+------------------+
|        1|            0|               0.0|
|        2|            3|               1.0|
|        3|            5|              1.25|
|        4|            7|               1.4|
|        5|            9|               1.5|
|        6|           11|1.5714285714285714|
|        7|           14|              1.75|
|        8|           16|1.7777777777777777|
+---------+-------------+------------------+
only showing top 8 rows


In [0]:
# ============================================================
# CELLULE 10 — RATIO COUPLE / VITESSE
#
# torque_speed_ratio = couple / (vitesse + 1)
# +1 pour éviter division par zéro si vitesse = 0
#
# Signal :
# Ratio trop haut = couple fort / vitesse faible → surcharge
# Ratio trop bas  = couple faible / vitesse haute → roue libre anormale
# ============================================================
df = df.withColumn(
    'torque_speed_ratio',
    F.col('torque_nm') / (F.col('rotation_speed_rpm') + 1)
)

print('✅ torque_speed_ratio créé')
df.select('sensor_id', 'torque_nm', 'rotation_speed_rpm', 'torque_speed_ratio') \
  .show(8)

✅ torque_speed_ratio créé


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+---------+------------------+--------------------+
|sensor_id|torque_nm|rotation_speed_rpm|  torque_speed_ratio|
+---------+---------+------------------+--------------------+
|        1|     42.8|            1551.0|0.027577319587628865|
|        2|     46.3|            1408.0| 0.03286018452803406|
|        3|     49.4|            1498.0| 0.03295530353569046|
|        4|     39.5|            1433.0|0.027545327754532774|
|        5|     40.0|            1408.0|0.028388928317955996|
|        6|     41.9|            1425.0|0.029382889200561008|
|        7|     42.4|            1558.0|0.027196921103271328|
|        8|     40.2|            1527.0|0.026308900523560212|
+---------+---------+------------------+--------------------+
only showing top 8 rows


In [0]:
# ============================================================
# CELLULE 11 — VARIABLE CIBLE ENRICHIE (failure_type)
#
# F.when() = équivalent du CASE WHEN en SQL
# On évalue les pannes dans un ordre précis
# .otherwise('NONE') = si aucune panne détectée
# ============================================================
df = df.withColumn(
    'failure_type',
    F.when(F.col('tool_wear_failure') == 1,        'TWF')
     .when(F.col('heat_dissipation_failure') == 1, 'HDF')
     .when(F.col('power_failure') == 1,            'PWF')
     .when(F.col('overstrain_failure') == 1,       'OSF')
     .when(F.col('random_failure') == 1,           'RNF')
     .otherwise('NONE')
)

print('✅ failure_type créé')
print('\nDistribution des types de pannes :')
df.groupBy('failure_type') \
  .count() \
  .orderBy('count', ascending=False) \
  .show()

✅ failure_type créé

Distribution des types de pannes :


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------------+-----+
|failure_type|count|
+------------+-----+
|        NONE| 9652|
|         HDF|  115|
|         PWF|   91|
|         OSF|   78|
|         TWF|   46|
|         RNF|   18|
+------------+-----+



In [0]:
# ============================================================
# CELLULE 12 — ANALYSE DÉSÉQUILIBRE DES CLASSES
#
# C'est un problème classique en maintenance prédictive :
# les pannes sont RARES → le dataset est très déséquilibré
# Ex : 97% NONE, 3% pannes → un modèle naïf prédit toujours NONE
# et atteint 97% d'accuracy sans jamais détecter une panne !
#
# Solution en J11 : SMOTE (sur-échantillonnage synthétique)
# ============================================================
total  = df.count()
pannes = df.filter(F.col('machine_failure') == 1).count()
sains  = total - pannes

print(f"{'='*45}")
print('ANALYSE DÉSÉQUILIBRE DES CLASSES')
print(f"{'='*45}")
print(f'Total lignes      : {total}')
print(f'Sans panne  (0)   : {sains}  ({round(sains/total*100, 1)}%)')
print(f'Avec panne  (1)   : {pannes} ({round(pannes/total*100, 1)}%)')
print(f'Ratio             : 1 panne pour {round(sains/pannes)} mesures normales')
print()
print('⚠️  Ce déséquilibre sera corrigé en J11 avec SMOTE')
print('    Un modèle entraîné sans correction donnera des résultats trompeurs')

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


ANALYSE DÉSÉQUILIBRE DES CLASSES
Total lignes      : 10000
Sans panne  (0)   : 9661  (96.6%)
Avec panne  (1)   : 339 (3.4%)
Ratio             : 1 panne pour 28 mesures normales

⚠️  Ce déséquilibre sera corrigé en J11 avec SMOTE
    Un modèle entraîné sans correction donnera des résultats trompeurs


In [0]:
# ============================================================
# CELLULE 13 — RAPPORT COMPLET DES FEATURES CRÉÉES
# ============================================================
features_silver    = 15  # colonnes après J5 (14 originales + timestamp)
features_finales   = len(df.columns)
features_creees    = features_finales - features_silver

print(f"{'='*45}")
print('RAPPORT FEATURE ENGINEERING')
print(f"{'='*45}")
print(f'Features Silver (entrée)  : {features_silver}')
print(f'Features créées           : {features_creees}')
print(f'Features totales (sortie) : {features_finales}')
print()
print('Liste complète des colonnes :')

nouvelles = [
    'rolling_mean_temp_1h', 'rolling_std_temp_1h',
    'rolling_mean_rotation_1h', 'rolling_std_rotation_1h',
    'rolling_mean_torque_1h', 'rolling_mean_torque_3h',
    'temp_diff', 'power_estimated_w',
    'tool_wear_rate', 'torque_speed_ratio',
    'failure_type'
]

for i, col in enumerate(df.columns, 1):
    tag = ' ← NOUVEAU' if col in nouvelles else ''
    print(f'  {i:2}. {col}{tag}')

RAPPORT FEATURE ENGINEERING
Features Silver (entrée)  : 15
Features créées           : 11
Features totales (sortie) : 26

Liste complète des colonnes :
   1. sensor_id
   2. product_id
   3. station_type
   4. air_temp_k
   5. process_temp_k
   6. rotation_speed_rpm
   7. torque_nm
   8. tool_wear_min
   9. machine_failure
  10. tool_wear_failure
  11. heat_dissipation_failure
  12. power_failure
  13. overstrain_failure
  14. random_failure
  15. timestamp
  16. rolling_mean_temp_1h ← NOUVEAU
  17. rolling_std_temp_1h ← NOUVEAU
  18. rolling_mean_rotation_1h ← NOUVEAU
  19. rolling_std_rotation_1h ← NOUVEAU
  20. rolling_mean_torque_1h ← NOUVEAU
  21. rolling_mean_torque_3h ← NOUVEAU
  22. temp_diff ← NOUVEAU
  23. power_estimated_w ← NOUVEAU
  24. tool_wear_rate ← NOUVEAU
  25. torque_speed_ratio ← NOUVEAU
  26. failure_type ← NOUVEAU


In [0]:
# ============================================================
# CELLULE 14 — VÉRIFICATION QUALITÉ AVANT ÉCRITURE
#
# On vérifie qu'aucune feature n'a généré de nulls inattendus
# Les rolling features ont des nulls normaux au début
# (pas assez de lignes précédentes pour calculer la fenêtre)
# → on les accepte, le modèle ML les gérera
# ============================================================
print('Vérification nulls sur les nouvelles features :')
nouvelles_cols = [
    'rolling_mean_temp_1h', 'rolling_std_temp_1h',
    'rolling_mean_rotation_1h', 'rolling_std_rotation_1h',
    'rolling_mean_torque_1h', 'rolling_mean_torque_3h',
    'temp_diff', 'power_estimated_w',
    'tool_wear_rate', 'torque_speed_ratio',
    'failure_type'
]

nulls = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in nouvelles_cols
]).collect()[0].asDict()

for col, n in nulls.items():
    statut = '✅' if n == 0 else '⚠️  (nulls en début de fenêtre — normal)'
    print(f'  {col:35s} : {n} nulls {statut}')

Vérification nulls sur les nouvelles features :


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


  rolling_mean_temp_1h                : 0 nulls ✅
  rolling_std_temp_1h                 : 1 nulls ⚠️  (nulls en début de fenêtre — normal)
  rolling_mean_rotation_1h            : 0 nulls ✅
  rolling_std_rotation_1h             : 1 nulls ⚠️  (nulls en début de fenêtre — normal)
  rolling_mean_torque_1h              : 0 nulls ✅
  rolling_mean_torque_3h              : 0 nulls ✅
  temp_diff                           : 0 nulls ✅
  power_estimated_w                   : 0 nulls ✅
  tool_wear_rate                      : 0 nulls ✅
  torque_speed_ratio                  : 0 nulls ✅
  failure_type                        : 0 nulls ✅


In [0]:
# ============================================================
# CELLULE 15 — ÉCRITURE SILVER ENRICHI (PATH_FEATURES)
#
# On écrit dans PATH_FEATURES (pas PATH_SILVER qu'on ne touche pas)
# Même technique que J5 : write → read pour remplacer le cache()
# ============================================================
df.write \
  .mode('overwrite') \
  .parquet(PATH_FEATURES)

print(f'✅ Silver enrichi écrit dans : {PATH_FEATURES}')

# Relecture immédiate pour confirmer
df = spark.read.parquet(PATH_FEATURES)
total_features = df.count()

print(f'   Lignes   : {total_features}')
print(f'   Colonnes : {len(df.columns)}')
print(f'\n🎉 J6 TERMINÉ — Feature Engineering OK !')
print('   Prêt pour J7 : rapport qualité Silver + export JSON')

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ Silver enrichi écrit dans : /Volumes/workspace/default/bronze/silver/sensor_features/
   Lignes   : 10000
   Colonnes : 26

🎉 J6 TERMINÉ — Feature Engineering OK !
   Prêt pour J7 : rapport qualité Silver + export JSON
